# Vietnamese Hate Speech Detection — Multi-Model Fine-tuning

**Dataset**: `data_final_sorted_cleaned_hsd.csv` — 21,497 ITViec reviews  
**Labels**: `CLEAN` · `OFFENSIVE` · `HATE`  
**GPU target**: NVIDIA L40S (48 GB VRAM)

---
## 🔁 Workflow — run one model per session

| Step | Action |
|------|--------|
| **1** | Go to the **`⚙️ CONFIG`** cell below and uncomment exactly **one** `MODEL_NAME` line |
| **2** | `Kernel → Restart Kernel and Run All Cells` |
| **3** | Wait for training to finish — F1 score is saved automatically to `experiment_history.json` |
| **4** | Repeat steps 1–3 for the next model |

The **Leaderboard** cell at the bottom always shows all completed runs ranked by F1 Macro.

---
## 🤖 Available models

| # | Model | Hub ID | Notes |
|---|-------|--------|-------|
| 1 | PhoBERT-large | `vinai/phobert-large` | Vietnamese RoBERTa, encoder-only |
| 2 | XLM-RoBERTa-large | `facebook/xlm-roberta-large` | Multilingual RoBERTa, encoder-only |
| 3 | mT5-large | `google/mt5-large` | Multilingual T5, encoder-decoder |
| 4 | ViT5-large | `VietAI/vit5-large` | Vietnamese T5, encoder-decoder |

In [ ]:
# ============================================================
# ⚙️  CONFIG — RTX 4090 · 24 GB · Max F1 build
# Edit this cell, then: Kernel → Restart & Run All
# ============================================================

# ── STEP 1: Uncomment exactly ONE model ─────────────────────
MODEL_NAME = "vinai/phobert-large"          # 1️⃣  PhoBERT-large        (Vietnamese RoBERTa)
# MODEL_NAME = "facebook/xlm-roberta-large" # 2️⃣  XLM-RoBERTa-large   (Multilingual RoBERTa)
# MODEL_NAME = "google/mt5-large"           # 3️⃣  mT5-large            (Multilingual T5)
# MODEL_NAME = "VietAI/vit5-large"          # 4️⃣  ViT5-large           (Vietnamese T5)

# ── STEP 2: Run mode ────────────────────────────────────────
SMOKE_TEST = False
GPU_MODE   = True    # RTX 4090 · Ada Lovelace sm_89 · bf16 + tf32

# ── STEP 3: Batch sizes — RTX 4090 (24 GB VRAM) ─────────────
# Effective batch = 16 × 2 = 32  (fair comparison across GPUs)
#
# GPU              VRAM    TRAIN_BATCH  GRAD_ACCUM  EVAL_BATCH
# RTX 4060 Ti      16 GB       8            4            16
# RTX 4090         24 GB      16            2            32     ← you are here
# L40S             48 GB      32            1            64
TRAIN_BATCH_SIZE  = 16
GRAD_ACCUM_STEPS  = 2
EVAL_BATCH_SIZE   = 32

# ── STEP 4: Training — tuned to maximise F1 Macro ───────────
MAX_LENGTH              = 256
NUM_TRAIN_EPOCHS        = 15   # minimum 15 as requested
LEARNING_RATE           = 2e-5
LR_SCHEDULER_TYPE       = "cosine"  # cosine decay outperforms linear for long runs
WARMUP_RATIO            = 0.05     # ~1 epoch warm-up over 15 epochs
WEIGHT_DECAY            = 0.05     # stronger regularisation for longer training
EARLY_STOPPING_PATIENCE = 0        # 0 = disabled → always train full 15 epochs
                                   # best checkpoint is still saved automatically
SEED                    = 42

# ── Experiment tracking ──────────────────────────────────────
USE_WANDB     = False
WANDB_PROJECT = "hsd-itviec"

# ── Paths ────────────────────────────────────────────────────
from pathlib import Path
try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).parent
except NameError:
    try:
        import ipynbname
        NOTEBOOK_DIR = ipynbname.path().parent
    except Exception:
        NOTEBOOK_DIR = Path().resolve()

DATA_PATH    = NOTEBOOK_DIR / "data_final_sorted_cleaned_hsd.csv"
HISTORY_PATH = NOTEBOOK_DIR / "experiment_history.json"
OUTPUT_DIR   = None

assert DATA_PATH.exists(), (
    f"\n❌ Dataset not found at: {DATA_PATH}\n"
    f"   Set NOTEBOOK_DIR manually to the folder containing the CSV."
)
print("✅ Config loaded")
print(f"   MODEL           : {MODEL_NAME}")
print(f"   Effective batch : {TRAIN_BATCH_SIZE} × {GRAD_ACCUM_STEPS} = {TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   Epochs          : {NUM_TRAIN_EPOCHS}")
print(f"   LR / Scheduler  : {LEARNING_RATE} / {LR_SCHEDULER_TYPE}")
print(f"   Early stopping  : {'disabled' if EARLY_STOPPING_PATIENCE == 0 else EARLY_STOPPING_PATIENCE}")


## 0 · GPU / environment check
Run this cell first to confirm CUDA is visible before spending time on data loading.

In [ ]:
import torch, sys, os
import transformers
from packaging.version import Version

tv  = Version(transformers.__version__)
ptv = Version(torch.__version__.split('+')[0])

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

# ── PyTorch version gate ─────────────────────────────────────
print()
if ptv < Version('2.6'):
    print('⚠️  PyTorch < 2.6 detected.')
    print('   torch.load is blocked due to CVE-2025-32434.')
    print('   Fix A (recommended): pip install torch --upgrade')
    print('   Fix B (workaround) : use_safetensors=True is already set in cell 8')
else:
    print(f'✅ PyTorch {torch.__version__} — CVE-2025-32434 patched natively')

# ── Transformers version gates ───────────────────────────────
issues = []
if tv < Version('4.37'):
    issues.append('use_cpu (smoke test)  needs transformers >= 4.37')
if tv < Version('4.34'):
    issues.append('save_only_model       needs transformers >= 4.34')
if tv < Version('4.46'):
    issues.append('processing_class=     needs transformers >= 4.46  (falls back to tokenizer= automatically)')
if issues:
    print()
    print('⚠️  Transformers compatibility notes:')
    for i in issues:
        print(f'   • {i}')
    print('   Run: pip install -U transformers  to resolve all at once')
else:
    print(f'✅ transformers {transformers.__version__} — all features supported')

# ── GPU info ─────────────────────────────────────────────────
if torch.cuda.is_available():
    print()
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram  = props.total_memory / 1024**3
        arch  = (
            'Blackwell'    if props.major >= 10 else
            'Ada Lovelace' if (props.major == 8 and props.minor >= 9) else
            'Ampere'       if props.major == 8 else 'Other'
        )
        print(f'  GPU  : {props.name}')
        print(f'  VRAM : {vram:.1f} GB')
        print(f'  Arch : sm_{props.major}{props.minor} ({arch})')
        print(f'  BF16 : {"✅" if props.major >= 8 else "❌"}')
        print(f'  TF32 : {"✅" if props.major >= 8 else "❌"}')
        if vram >= 20:
            print(f'  ✅  {vram:.0f} GB — batch={TRAIN_BATCH_SIZE}×{GRAD_ACCUM_STEPS}={TRAIN_BATCH_SIZE*GRAD_ACCUM_STEPS} safe for all 4 models')
        elif vram >= 14:
            print(f'  ⚠️  {vram:.0f} GB — reduce TRAIN_BATCH_SIZE=8 GRAD_ACCUM_STEPS=4')
        else:
            print(f'  ❌  {vram:.0f} GB — reduce TRAIN_BATCH_SIZE=4')
elif GPU_MODE:
    raise RuntimeError('GPU_MODE=True but no CUDA GPU found.')
else:
    print('No GPU — CPU mode.')


## 1 · Install dependencies
Skip this cell if your environment already has the packages.

In [ ]:
# Uncomment and run once if needed
# !pip install transformers datasets evaluate scikit-learn wandb accelerate torch --quiet

## 2 · Imports & logging

In [ ]:
from __future__ import annotations

import json
import logging
import os
import re
from datetime import datetime, timezone
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
import evaluate

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

set_seed(SEED)
print("✅ Imports OK")

## 3 · Constants & label mapping

In [ ]:
LABEL2ID: Dict[str, int] = {"CLEAN": 0, "OFFENSIVE": 1, "HATE": 2}
ID2LABEL: Dict[int, str] = {v: k for k, v in LABEL2ID.items()}

TEXT_COLUMN  = "review_content_cleaned"
LABEL_COLUMN = "category"

# Resolve output directory from config
def model_slug(model_name: str) -> str:
    """Convert 'vinai/phobert-large' → 'phobert-large'."""
    slug = model_name.split("/")[-1]
    return re.sub(r"[^a-zA-Z0-9_\-]", "-", slug)

slug = model_slug(MODEL_NAME)
output_dir: Path = OUTPUT_DIR or (NOTEBOOK_DIR / "saved_models" / f"{slug}-hsd")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Model      : {MODEL_NAME}")
print(f"Slug       : {slug}")
print(f"Output dir : {output_dir}")
print(f"Smoke test : {SMOKE_TEST}")
print(f"GPU mode   : {GPU_MODE}")

## 4 · Weights & Biases setup

In [ ]:
report_to: List[str] = []

if USE_WANDB:
    try:
        import wandb
        run_name = f"{slug}-{'smoke' if SMOKE_TEST else 'full'}"
        os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)
        report_to = ["wandb"]
        logger.info("W&B enabled — project: %s | run: %s", WANDB_PROJECT, run_name)
    except ImportError:
        logger.warning("wandb not installed — disabling. Run: pip install wandb")
else:
    os.environ["WANDB_DISABLED"] = "true"
    logger.info("W&B disabled.")

print(f"report_to: {report_to if report_to else 'none (local only)'}")

## 5 · Load & split dataset

Stratified **80 / 10 / 10** split (train / val / test).

In [ ]:
def load_and_split(
    data_path: Path,
    smoke_test: bool,
    seed: int,
):
    """Load CSV and return train / val / test DataFrames via stratified split."""
    logger.info("Loading dataset from %s", data_path)
    df = pd.read_csv(data_path)

    # Clean
    df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
    df[LABEL_COLUMN] = df[LABEL_COLUMN].str.strip().str.upper()
    df = df[df[LABEL_COLUMN].isin(LABEL2ID.keys())].reset_index(drop=True)

    logger.info(
        "Dataset size after cleaning: %d  |  Distribution: %s",
        len(df), df[LABEL_COLUMN].value_counts().to_dict(),
    )

    if smoke_test:
        df = df.sample(n=min(1_000, len(df)), random_state=seed).reset_index(drop=True)
        logger.warning("SMOKE TEST: using %d samples.", len(df))

    # Stratified 80 / 10 / 10
    train_df, temp_df = train_test_split(
        df, test_size=0.20, stratify=df[LABEL_COLUMN], random_state=seed
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df[LABEL_COLUMN], random_state=seed
    )

    logger.info(
        "Split sizes — train: %d | val: %d | test: %d",
        len(train_df), len(val_df), len(test_df),
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


train_df, val_df, test_df = load_and_split(DATA_PATH, SMOKE_TEST, SEED)

# Quick visual check
print("\nLabel distribution per split:")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df[LABEL_COLUMN].value_counts().to_dict()
    print(f"  {name:5s} ({len(df):6,}): {counts}")

## 6 · Convert to HuggingFace DatasetDict

In [ ]:
def df_to_hf_dataset(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> DatasetDict:
    """Convert pandas DataFrames to a HuggingFace DatasetDict."""

    def _convert(df: pd.DataFrame) -> Dataset:
        return Dataset.from_dict(
            {
                "text":  df[TEXT_COLUMN].tolist(),
                "label": [LABEL2ID[lbl] for lbl in df[LABEL_COLUMN]],
            }
        )

    return DatasetDict(
        {
            "train":      _convert(train_df),
            "validation": _convert(val_df),
            "test":       _convert(test_df),
        }
    )


raw_datasets = df_to_hf_dataset(train_df, val_df, test_df)
print(raw_datasets)

## 7 · Load tokenizer & tokenise dataset

> **⚠️ PhoBERT note** — `vinai/phobert-*` models were pre-trained on VnCoreNLP word-segmented text.
> For best accuracy, pre-segment with `py_vncorenlp` before tokenisation:
> ```python
> from py_vncorenlp import VnCoreNLP
> rdrsegmenter = VnCoreNLP(annotators=["wseg"])
> text = " ".join(rdrsegmenter.word_segment(text))
> ```
> XLM-RoBERTa and SentencePiece-based models do **not** require this step.

In [ ]:
logger.info("Loading tokenizer: %s", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# T5-family fix: mt5 / vit5 tokenizers ship without a pad_token.
# DataCollatorWithPadding crashes if pad_token is None — fix it here.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    logger.warning(
        "Tokenizer has no pad_token (T5-family model). "
        "Setting pad_token = eos_token ('%s').", tokenizer.eos_token
    )

print(f"Tokenizer type : {type(tokenizer).__name__}")
print(f"Vocab size     : {tokenizer.vocab_size:,}")
print(f"Pad token      : '{tokenizer.pad_token}'")


In [ ]:
def tokenize_dataset(
    dataset: DatasetDict,
    tokenizer: AutoTokenizer,
    max_length: int,
) -> DatasetDict:
    """Batch-tokenise all splits with dynamic padding."""
    logger.info("Tokenising dataset (max_length=%d)…", max_length)

    def _tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length,
            padding=False,   # dynamic padding via DataCollatorWithPadding
        )

    tokenized = dataset.map(
        _tokenize,
        batched=True,
        remove_columns=["text"],
        desc="Tokenising",
    )
    tokenized.set_format("torch")
    return tokenized


tokenized_datasets = tokenize_dataset(raw_datasets, tokenizer, MAX_LENGTH)
print(tokenized_datasets)

## 8 · Load model

The classification head (`classifier.*`) is randomly initialised — this is expected.  
The `UNEXPECTED` keys in the LOAD REPORT are the pre-training LM head being discarded, which is also normal.

In [ ]:
logger.info('Loading model: %s', MODEL_NAME)

# use_safetensors=True — loads the .safetensors file instead of .bin
# Best practice regardless of torch version:
#   • torch >= 2.6 : CVE-2025-32434 is patched natively, both formats work
#   • torch <  2.6 : required — .bin loading is blocked by transformers
# All 4 hub models (phobert-large, xlm-roberta-large, mt5-large, vit5-large)
# publish safetensors files so this is always safe to keep on.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    use_safetensors=True,
)

# Sync pad_token_id for T5-family models (mt5, vit5)
if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')
print(f'pad_token_id     : {model.config.pad_token_id}')


## 9 · Metrics — F1 Macro + Accuracy

In [ ]:
def build_compute_metrics():
    """Return a closure that computes Macro-F1 and accuracy."""
    f1_metric  = evaluate.load("f1")
    acc_metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        f1  = f1_metric.compute(predictions=predictions, references=labels, average="macro")
        acc = acc_metric.compute(predictions=predictions, references=labels)
        return {"f1_macro": f1["f1"], "accuracy": acc["accuracy"]}

    return compute_metrics

print("✅ compute_metrics ready")

## 10 · Training arguments

Mode is selected automatically from `SMOKE_TEST` / `GPU_MODE` flags in the CONFIG cell.

In [ ]:
import transformers
from packaging.version import Version

# save_only_model was added in transformers 4.34
# Gracefully omit it on older installations rather than crashing
_supports_save_only_model = Version(transformers.__version__) >= Version("4.34")
_supports_processing_class = Version(transformers.__version__) >= Version("4.46")

if not _supports_save_only_model:
    print("⚠️  save_only_model not supported on this transformers version — skipping")
if not _supports_processing_class:
    print("⚠️  processing_class not supported — will fall back to tokenizer= argument")


def build_training_args(
    output_dir: Path,
    smoke_test: bool,
    gpu_mode: bool,
    train_batch_size: int,
    eval_batch_size: int,
    grad_accum_steps: int,
    num_train_epochs: int,
    learning_rate: float,
    lr_scheduler_type: str,
    warmup_ratio: float,
    weight_decay: float,
    report_to: List[str],
) -> TrainingArguments:
    """TrainingArguments — max-F1 build for RTX 4090 / 15+ epochs."""

    common = dict(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,           # keep only the single best checkpoint
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=20,
        report_to=report_to,
        seed=SEED,
    )

    # ── Smoke test ─────────────────────────────────────────────
    if smoke_test:
        return TrainingArguments(
            **common,
            num_train_epochs=1,
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=1,
            fp16=False, bf16=False,
            use_cpu=True,
        )

    # ── GPU run ────────────────────────────────────────────────
    if gpu_mode:
        gpu_kwargs = dict(
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=train_batch_size,
            per_device_eval_batch_size=eval_batch_size,
            gradient_accumulation_steps=grad_accum_steps,
            learning_rate=learning_rate,
            lr_scheduler_type=lr_scheduler_type,
            warmup_ratio=warmup_ratio,
            weight_decay=weight_decay,
            bf16=True,
            tf32=True,
            fp16=False,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
        )
        # Only pass save_only_model on transformers >= 4.34
        if _supports_save_only_model:
            gpu_kwargs["save_only_model"] = True
        return TrainingArguments(**common, **gpu_kwargs)

    # ── CPU fallback ───────────────────────────────────────────
    return TrainingArguments(
        **common,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=grad_accum_steps,
        learning_rate=learning_rate,
        lr_scheduler_type=lr_scheduler_type,
        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        fp16=False,
    )


training_args = build_training_args(
    output_dir=output_dir,
    smoke_test=SMOKE_TEST,
    gpu_mode=GPU_MODE,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    report_to=report_to,
)

eff = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
mode = "smoke_test" if SMOKE_TEST else ("GPU" if GPU_MODE else "CPU")
print(f"Mode             : {mode}")
print(f"Epochs           : {training_args.num_train_epochs}")
print(f"Train batch      : {training_args.per_device_train_batch_size} × "
      f"{training_args.gradient_accumulation_steps} = {eff} effective")
print(f"Eval batch       : {training_args.per_device_eval_batch_size}")
print(f"LR / Scheduler   : {training_args.learning_rate} / {training_args.lr_scheduler_type}")
print(f"Weight decay     : {training_args.weight_decay}")
print(f"bf16 / tf32      : {training_args.bf16} / {training_args.tf32}")
print(f"save_total_limit : {training_args.save_total_limit}  (best only)")
print(f"save_only_model  : {_supports_save_only_model}")


## 11 · Build Trainer

In [ ]:
callbacks = []
if EARLY_STOPPING_PATIENCE > 0 and not SMOKE_TEST:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE))
    print(f"Early stopping   : patience={EARLY_STOPPING_PATIENCE} epochs")
else:
    print("Early stopping   : disabled — full epochs will run, best checkpoint kept")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# processing_class= replaces tokenizer= in transformers >= 4.46
# Fall back to tokenizer= for older versions (e.g. 4.34–4.45)
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(),
    callbacks=callbacks,
)
if _supports_processing_class:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

total_steps = (
    len(tokenized_datasets["train"])
    // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
    * training_args.num_train_epochs
)
print(f"\n✅ Trainer ready")
print(f"   Total optimiser steps : ~{total_steps:,}")
print(f"   Warmup steps          : ~{int(total_steps * training_args.warmup_ratio):,}")


## 12 · Train 🚀

Progress bars and per-epoch metrics will appear below.

In [ ]:
logger.info("=" * 60)
logger.info("Starting training — model: %s", MODEL_NAME)
logger.info("  smoke_test=%s | gpu_mode=%s | epochs=%d",
            SMOKE_TEST, GPU_MODE, training_args.num_train_epochs)
logger.info("=" * 60)

train_result = trainer.train()

## 13 · Save best model & tokenizer

In [ ]:
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))

print(f"✅ Model + tokenizer saved to: {output_dir}")
print("Files saved:")
for f in sorted(output_dir.iterdir()):
    print(f"  {f.name}")

## 14 · Evaluate on held-out test set

In [ ]:
logger.info("Evaluating on test set…")
test_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["test"],
    metric_key_prefix="test",
)

print("\n📊 Test set results:")
for k, v in test_metrics.items():
    print(f"  {k:<30} {v:.6f}" if isinstance(v, float) else f"  {k:<30} {v}")

In [ ]:
# Also evaluate validation set — used as the canonical score for the leaderboard
val_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["validation"],
    metric_key_prefix="eval",
)

f1_macro  = val_metrics.get("eval_f1_macro",  test_metrics.get("test_f1_macro",  0.0))
eval_loss = val_metrics.get("eval_loss",       test_metrics.get("test_loss",      0.0))

print(f"Val F1 Macro : {f1_macro:.4f}")
print(f"Val Loss     : {eval_loss:.4f}")

## 15 · Update experiment leaderboard (`experiment_history.json`)

In [ ]:
def update_experiment_history(
    history_path: Path,
    model_name: str,
    f1_macro: float,
    eval_loss: float,
    output_dir: Path,
    extra: Optional[Dict] = None,
) -> None:
    """
    Append one run's result to experiment_history.json.
    File is kept sorted by f1_macro descending — best run always at top.
    """
    history: List[Dict] = []
    if history_path.exists():
        try:
            with open(history_path, "r", encoding="utf-8") as fh:
                history = json.load(fh)
        except json.JSONDecodeError:
            logger.warning("experiment_history.json is malformed — starting fresh.")

    entry = {
        "model_name":  model_name,
        "timestamp":   datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "f1_macro":    round(f1_macro, 6),
        "eval_loss":   round(eval_loss, 6),
        "saved_model": str(output_dir),
        **(extra or {}),
    }
    history.append(entry)
    history.sort(key=lambda r: r.get("f1_macro", 0.0), reverse=True)

    with open(history_path, "w", encoding="utf-8") as fh:
        json.dump(history, fh, indent=2, ensure_ascii=False)

    logger.info("Experiment history updated → %s", history_path)
    return history


history = update_experiment_history(
    history_path=HISTORY_PATH,
    model_name=MODEL_NAME,
    f1_macro=f1_macro,
    eval_loss=eval_loss,
    output_dir=output_dir,
    extra={
        "test_f1_macro":  test_metrics.get("test_f1_macro"),
        "test_loss":      test_metrics.get("test_loss"),
        "train_runtime":  train_result.metrics.get("train_runtime"),
        "train_samples":  len(tokenized_datasets["train"]),
        "max_length":     MAX_LENGTH,
        "epochs":         training_args.num_train_epochs,
        "batch_size":     training_args.per_device_train_batch_size,
        "learning_rate":  LEARNING_RATE,
        "smoke_test":     SMOKE_TEST,
        "gpu_mode":       GPU_MODE,
        "seed":           SEED,
    },
)

print(f"✅ Saved to {HISTORY_PATH}")

## 16 · Leaderboard 🏆

In [ ]:
import json

ALL_MODELS = [
    "vinai/phobert-large",
    "facebook/xlm-roberta-large",
    "google/mt5-large",
    "VietAI/vit5-large",
]

# Load history (may not exist yet on first run)
history = []
if HISTORY_PATH.exists():
    with open(HISTORY_PATH) as f:
        history = json.load(f)

trained = {r["model_name"] for r in history}
pending = [m for m in ALL_MODELS if m not in trained]

print("=" * 90)
print("🏆  LEADERBOARD")
print("=" * 90)
print(f"{'Rank':<6} {'Model':<45} {'F1 Macro':>10} {'Eval Loss':>10}  Timestamp")
print("-" * 90)
for rank, run in enumerate(history, 1):
    tag = " [smoke]" if run.get("smoke_test") else ""
    print(
        f"#{rank:<5} {(run['model_name'] + tag):<45}"
        f" {run['f1_macro']:>10.4f} {run['eval_loss']:>10.4f}"
        f"  {run['timestamp']}"
    )

if pending:
    print()
    print("⏳  Not yet trained:")
    for m in pending:
        print(f"   • {m}")
    print()
    next_model = pending[0]
    print(f"👉  Next up → set MODEL_NAME = \"{next_model}\" in the CONFIG cell, then Restart & Run All.")
else:
    print()
    print("✅  All 4 models trained! Check experiment_history.json for full details.")

print("=" * 90)
print(f"\n🏁  This run → {MODEL_NAME}  saved to: {output_dir}")
